# Multi-dataset: Differential Expression Analysis

This example demonstrates how to identify **differentially expressed genes** in a multi-FOV
setting using `spatial_query_multi.de_genes`.

Two common comparisons are shown:

1. **Across conditions:** Compare motif+ anchor cells between two conditions
   (e.g., motif+ B cells in CLR vs DII)
2. **Within a condition:** Compare motif+ vs motif− anchor cells within one condition

**Dataset:** CODEX spatial proteomics — CLR vs DII immune subtypes
**Key API:** `spatial_query_multi.de_genes`

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import numpy as np

from SpatialQuery import spatial_query_multi

## Load Data & Initialize

In [4]:
DATA_DIR = "../data/codex_cancer"

adata = ad.read_h5ad(f"{DATA_DIR}/codex_data.h5ad")

adata

AnnData object with n_obs × n_vars = 258385 × 56
    obs: 'CellID', 'ClusterID', 'EventID', 'File Name', 'Region', 'TMA_AB', 'TMA_12', 'Index in File', 'groups', 'patients', 'spots', 'cell_id', 'size:size', 'HOECHST1_Cyc_1_ch_1', 'DRAQ5_Cyc_23_ch_4', 'Profile_Homogeneity:Fiter1', 'ClusterSize', 'ClusterName', 'neighborhood10', 'CD4+ICOS+', 'CD4+Ki67+', 'CD4+PD-1+', 'CD68+CD163+ICOS+', 'CD68+CD163+Ki67+', 'CD68+CD163+PD-1+', 'CD68+ICOS+', 'CD68+Ki67+', 'CD68+PD-1+', 'CD8+ICOS+', 'CD8+Ki67+', 'CD8+PD-1+', 'Treg-ICOS+', 'Treg-Ki67+', 'Treg-PD-1+', 'neighborhood number final', 'neighborhood name'
    var: 'marker_name', 'full_name', 'cell_type_annotation', 'cycle', 'channel'
    uns: 'data_source', 'groups_mapping', 'n_markers', 'technology'
    obsm: 'X_spatial_global', 'X_spatial_tile'

In [5]:
adata.X.max()

np.float32(54776.695)

Inspect `adata.obsm` for spatial coordinates, `adata.obs` for cell type labels and FOV identifiers, and `adata.var` for feature names. Then set the corresponding column names below.

In [6]:
# Assign immune subtype labels
adata.obs["state"] = "CLR"
adata.obs["state"][adata.obs["groups"] == 2] = "DII"

# Feature-wise z-score normalization for protein expression
adata.X = (adata.X - adata.X.mean(axis=0)) / adata.X.std(axis=0)


In [7]:
spatial_key = "X_spatial_tile"
label_key = "ClusterName"
feature_name = "marker_name"
fov_key = "File Name"

In [8]:
# Split data by FOV and immune subtype
clr_data = adata[adata.obs["state"] == "CLR"]
dii_data = adata[adata.obs["state"] == "DII"]

clr_datas = [clr_data[clr_data.obs[fov_key] == f] for f in clr_data.obs[fov_key].unique()]
dii_datas = [dii_data[dii_data.obs[fov_key] == f] for f in dii_data.obs[fov_key].unique()]

ds_names = ["CLR"] * len(clr_datas) + ["DII"] * len(dii_datas)
adata_fovs = clr_datas + dii_datas

print(f"CLR FOVs: {len(clr_datas)}, DII FOVs: {len(dii_datas)}")

CLR FOVs: 68, DII FOVs: 72


In [9]:
spm = spatial_query_multi(
    adatas=adata_fovs,
    datasets=ds_names,
    spatial_key=spatial_key,
    label_key=label_key,
    feature_name=feature_name,
    build_gene_index=False,
    if_lognorm=False,  # data is already z-score normalized
    if_normalize_spatial_coord=True,
)


Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0448
build_gene_index is False. Using adata.X for gene expression analysis.

Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0311
build_gene_index is False. Using adata.X for gene expression analysis.

Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0488
build_gene_index is False. Using adata.X for gene expression analysis.

Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0587
build_gene_index is False. Using adata.X for gene expression analysis.

Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0603
build_gene_index is False. Using adata.X for gene expression analysis.

Auto-normalizing spatial coordinates: mean nearest neighbor distance = 1.0
Scale factor: 0.0432
build_gene_index is False. Using adata.X for gene expression an

## Define Anchor and Motif

We use **B cells** as the anchor and test a TLS-associated motif.

In [10]:
anchor_ct = "B cells"
max_dist = 5
clr = "CLR"
dii = "DII" 

In [11]:
motif = ["B cells", "CD4+ T cells CD45RO+", "CD8+ T cells"]

## Retrieve Motif+ Cell IDs Per Condition

Use `motif_enrichment_dist` with `return_cellID=True` to obtain the indices of motif-positive
anchor cells and their neighbors for each condition. The returned cell IDs are organized
as dictionaries keyed by FOV name.

> **Note:** In the multi-FOV setting, `de_genes` expects `ind_group1` and `ind_group2`
> to be dictionaries `{fov_name: [cell_indices]}`, not flat lists.

In [12]:
clr_result, clr_motif_id, clr_center_id = spm.motif_enrichment_dist(
    ct=anchor_ct, motifs=motif, dataset=clr, max_dist=max_dist, return_cellID=True,
)

dii_result, dii_motif_id, dii_center_id = spm.motif_enrichment_dist(
    ct=anchor_ct, motifs=motif, dataset=dii, max_dist=max_dist, return_cellID=True,
)

print(f"CLR motif+ anchors: {sum(len(v) for v in clr_center_id[str(sorted(motif))].values())} cells")
print(f"DII motif+ anchors: {sum(len(v) for v in dii_center_id[str(sorted(motif))].values())} cells")

CLR motif+ anchors: 6680 cells
DII motif+ anchors: 1336 cells


### Alternative: KNN-based Cell ID Retrieval

You can also retrieve motif-positive cell IDs using KNN neighborhoods.

In [13]:
k = 30

clr_result_knn, clr_motif_id_knn, clr_center_id_knn = spm.motif_enrichment_knn(
    ct=anchor_ct, motifs=motif, dataset=clr, k=k, return_cellID=True,
)

dii_result_knn, dii_motif_id_knn, dii_center_id_knn = spm.motif_enrichment_knn(
    ct=anchor_ct, motifs=motif, dataset=dii, k=k, return_cellID=True,
)

print(f"KNN CLR motif+ anchors: {sum(len(v) for v in clr_center_id_knn[str(sorted(motif))].values())} cells")
print(f"KNN DII motif+ anchors: {sum(len(v) for v in dii_center_id_knn[str(sorted(motif))].values())} cells")

KNN CLR motif+ anchors: 6080 cells
KNN DII motif+ anchors: 1180 cells


## Comparison 1: Motif+ Cells Across Conditions

Compare gene expression of motif-positive anchor cells between CLR and DII.

**Key parameters (multi-FOV version):**

| Parameter | Description |
|-----------|-------------|
| `ind_group1` | Dict of `{fov_name: [cell_indices]}` for group 1 |
| `ind_group2` | Dict of `{fov_name: [cell_indices]}` for group 2 |
| `method` | Statistical test: `'t-test'`, `'wilcoxon'`, or `'fisher'` |
| `alpha` | Significance threshold |

The multi-FOV `de_genes` automatically handles pooling across FOVs.

In [14]:
de_across = spm.de_genes(
    ind_group1=clr_center_id[str(sorted(motif))],
    ind_group2=dii_center_id[str(sorted(motif))],
    method="t-test",
    alpha=0.05,
)

print(f"DE genes (CLR vs DII motif+ {anchor_ct}): {len(de_across)}")
de_across.head(10)

Testing 53 genes ...
DE genes (CLR vs DII motif+ B cells): 49


,gene,p_value,adj-pval,log2fc,proportion_1,proportion_2,abs_difference,de_in
0,beta-catenin,1.204263e-96,6.382593e-95,0.958576,1.0,1.0,0.0,group1
1,Podoplanin,3.765711e-94,6.815516e-93,NaN,1.0,1.0,0.0,None
2,CD56,3.857839e-94,6.815516e-93,0.562067,1.0,1.0,0.0,group1
3,LAG-3,1.500285e-64,1.987878e-63,4.336867,1.0,1.0,0.0,group1
4,VISTA,1.754321e-49,1.859580e-48,1.029951,1.0,1.0,0.0,group1
5,MMP12,7.135779e-48,6.303272e-47,3.534385,1.0,1.0,0.0,group1
6,CD68,5.081177e-43,3.847177e-42,0.320779,1.0,1.0,0.0,group1
7,CDX2,6.467689e-41,4.284844e-40,-3.627088,1.0,1.0,0.0,group2
8,CD38,1.582104e-36,9.316835e-36,6.362736,1.0,1.0,0.0,group1
9,aSMA,7.876314e-35,4.174446e-34,0.426265,1.0,1.0,0.0,group1


## Comparison 2: Motif+ vs Motif− Within DII

Compare gene expression between motif-positive and motif-negative anchor cells
within the DII condition only.

In [15]:
# Get non-motif anchor cell IDs for DII FOVs
non_motif_center = {str(sorted(motif)): {}}
for sp in spm.spatial_queries:
    if sp.dataset.split("_")[0] != dii:
        continue
    ct_id = np.where(sp.labels == anchor_ct)[0]
    motif_ids = dii_center_id[str(sorted(motif))].get(sp.dataset, [])
    non_motif_center[str(sorted(motif))][sp.dataset] = list(set(ct_id) - set(motif_ids))

In [16]:
de_within = spm.de_genes(
    ind_group1=dii_center_id[str(sorted(motif))],
    ind_group2=non_motif_center[str(sorted(motif))],
    method="t-test",
    alpha=0.05,
)

print(f"DE genes (motif+ vs motif− in DII): {len(de_within)}")
de_within.head(10)

Testing 56 genes ...
DE genes (motif+ vs motif− in DII): 14


,gene,p_value,adj-pval,log2fc,proportion_1,proportion_2,abs_difference,de_in
0,CD30,1.116427e-07,0.000006,-2.945715,1.0,1.0,0.0,group2
1,LAG-3,6.494062e-06,0.000182,NaN,1.0,1.0,0.0,None
2,CD11c,1.072207e-04,0.001583,NaN,1.0,1.0,0.0,None
3,Podoplanin,1.130977e-04,0.001583,0.710304,1.0,1.0,0.0,group1
4,CD163,1.955508e-04,0.002190,0.093056,1.0,1.0,0.0,group1
5,BCL-2,4.291030e-04,0.004005,-0.676935,1.0,1.0,0.0,group2
6,CD20,6.402517e-04,0.005122,1.049646,1.0,1.0,0.0,group1
7,GATA3,7.641967e-04,0.005141,-1.975538,1.0,1.0,0.0,group2
8,HLA-DR,8.262002e-04,0.005141,2.009964,1.0,1.0,0.0,group1
9,CD45,1.936932e-03,0.010847,0.415165,1.0,1.0,0.0,group1
